In [32]:
from transformers import BertForSequenceClassification, BertTokenizer, BertTokenizerFast

MODEL_NAME = "bert-base-uncased"

try:
    tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)
    tokenizer_mode = "BertTokenizerFast"
except Exception:
    tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
    tokenizer_mode = "BertTokenizer"

model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)

print(f"Tokenizer loaded with {tokenizer_mode}; vocab size = {tokenizer.vocab_size}")
print(f"Model loaded: {model.__class__.__name__}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Tokenizer loaded with BertTokenizerFast; vocab size = 30522
Model loaded: BertForSequenceClassification


In [33]:
import gc
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/tommaso/Documenti/SUPSI/BACHELOR/3_anno_bech/primaverile/a/M-P6203E-DataProjects-Hackaton3_P1")\nif not (PROJECT_ROOT / "utils").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "utils").exists():
PROJECT_ROOT = Path("/home/tommaso/Documenti/SUPSI/BACHELOR/3_anno_bech/primaverile/a/M-P6203E-DataProjects-Hackaton3_P1")\n            break
sys.path.append(str(PROJECT_ROOT))

In [26]:
import pandas as pd

df_train = pd.read_parquet(PROJECT_ROOT / "data/exploded_splits/train_pairs.parquet")
df_val = pd.read_parquet(PROJECT_ROOT / "data/exploded_splits/validation_pairs.parquet")
df_test = pd.read_parquet(PROJECT_ROOT / "data/exploded_splits/test_pairs.parquet")

df_train = df_train.sample(frac=1.0, random_state=42, ignore_index=True)
df_val = df_val.sample(frac=1.0, random_state=42, ignore_index=True)
df_test = df_test.sample(frac=1.0, random_state=42, ignore_index=True)

print("Train size:", len(df_train))
print("Validation size:", len(df_val))
print("Test size:", len(df_test))

Train size: 2162520
Validation size: 391242
Test size: 396382


In [ ]:
import gc
import torch
from torch.backends import cudnn

from utils.training import (
    PlotLossCallback,
    WeightedTrainer,
    compute_pos_weight,
    create_training_arguments,
    get_device,
    set_seed,
)

from utils.citation_dataset import BertCitationDataset

# -----------------------------
# 4) BERT training
# -----------------------------

set_seed(42)
cudnn.benchmark = True
device = get_device()
print(f"Active device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

training_args = create_training_arguments(
    output_dir="./bert_citation",
    per_device_train_batch_size=96,
    per_device_eval_batch_size=64,
    num_train_epochs=1,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=20,
    save_total_limit=2,
    report_to=[],
)


Active device: cuda
GPU: NVIDIA GeForce RTX 3060
Building datasets...


In [34]:
df_name = "exploded_splits"
model_family = "transformer"
model_name = "bert_sequence_classifier"
split_name = "predefined_train_validation"

final_path = PROJECT_ROOT / "Models" / 'embeddings' / 'bert_citation'

model = model.to(device)
loss_callback = PlotLossCallback(save_dir=final_path)

In [ ]:
print("Building datasets...")
train_dataset = BertCitationDataset(df_train, tokenizer, max_len=128, include_authors=True)
val_dataset = BertCitationDataset(df_val, tokenizer, max_len=128, include_authors=True)

In [35]:
# Carica l'ultimo checkpoint BERT se esiste
from pathlib import Path
import os

def find_latest_checkpoint(root: Path) -> Path | None:
    # 1) Check common output dir used in training
    bert_out = root / "bert_citation"
    if bert_out.exists():
        # prefer checkpoint-* subfolders, otherwise the folder itself
        subdirs = [d for d in bert_out.iterdir() if d.is_dir()]
        if subdirs:
            return sorted(subdirs, key=lambda p: p.stat().st_mtime, reverse=True)[0]
        return bert_out

    # 2) Search `models/` for folders containing a saved model file
    models_root = root / "models"
    if models_root.exists():
        candidates = []
        for p in models_root.rglob("*"):
            if p.is_dir():
                if (p / "pytorch_model.bin").exists() or (p / "tf_model.h5").exists() or (p / "config.json").exists():
                    candidates.append(p)
        if candidates:
            return sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)[0]

    return None

ckpt_dir = find_latest_checkpoint(PROJECT_ROOT / "Models" / 'embeddings')
if ckpt_dir is not None:
    try:
        print(f"Found checkpoint directory: {ckpt_dir}. Attempting to load...")
        from transformers import BertForSequenceClassification

        model = BertForSequenceClassification.from_pretrained(str(ckpt_dir), num_labels=1)
        print("Checkpoint loaded into `model`.")
    except Exception as e:
        print("Failed to load checkpoint:", e)
else:
    print("No checkpoint found. Using freshly initialized model.")

Found checkpoint directory: /home/tommaso/Documenti/SUPSI/BACHELOR/3_anno_bech/primaverile/M-P6203E-DataProjects-Hackaton3_P1/Models/embeddings/bert_citation/checkpoint-4500. Attempting to load...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Checkpoint loaded into `model`.


In [36]:
pos_weight = compute_pos_weight(df_train["is_reference_valid"].to_numpy()).to(device)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    pos_weight=pos_weight,
    callbacks=[loss_callback],
)

print("Starting training...")
trainer.train()
model = trainer.model

print("\nTraining completed.")

Starting training...


Step,Training Loss
20,0.126357


KeyboardInterrupt: 

In [ ]:
from utils.model_saver import save_model_artifact

df_name = "embeddings/exploded_splits"

model_dir, summary_path = save_model_artifact(
    trainer.model,
    df_name=df_name,
    model_family=model_family,
    model_name=model_name,
    split_name=split_name,
    params={
        "train_size": len(df_train),
        "val_size": len(df_val),
    },
    tokenizer=tokenizer,
    summary={
        "train_size": len(df_train),
        "val_size": len(df_val),
    },
    force=True,
)
print("Saved transformer model to:", model_dir)
print("Saved summary to:", summary_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved transformer model to: /home/tommaso/Documenti/SUPSI/BACHELOR/3_anno_bech/primaverile/M-P6203E-DataProjects-Hackaton3_P1/Models/exploded_splits/transformer/bert_sequence_classifier/predefined_train_validation/bert_sequence_cl__train_size-2162520__val_size-391242__7a9774a2dce8
Saved summary to: /home/tommaso/Documenti/SUPSI/BACHELOR/3_anno_bech/primaverile/M-P6203E-DataProjects-Hackaton3_P1/Models/exploded_splits/transformer/bert_sequence_classifier/predefined_train_validation/summary__bert_sequence_cl__20260423T134126.json


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Subset

from utils.training import evaluate_predictions

# -----------------------------
# 6) Validation metrics and confusion matrix
# -----------------------------

offset = 0
max_eval_samples = 5000

# Support both Subset (has .dataset/.indices) and plain Dataset
if max_eval_samples:
    if hasattr(val_dataset, "dataset") and hasattr(val_dataset, "indices"):
        base_dataset = val_dataset.dataset
        indices = val_dataset.indices[offset : offset + max_eval_samples]
    else:
        base_dataset = val_dataset
        indices = list(range(offset, min(len(val_dataset), offset + max_eval_samples)))
    eval_subset = Subset(base_dataset, indices)
else:
    eval_subset = val_dataset

pred_output = trainer.predict(eval_subset)

logits = pred_output.predictions.reshape(-1)
if logits.ndim > 1:
    logits = logits.squeeze()
y_prob = 1 / (1 + np.exp(-logits))
y_true = pred_output.label_ids.astype(int).reshape(-1)

metrics = evaluate_predictions(y_true, y_prob, threshold=0.5)
y_pred = metrics["y_pred"]
cm = metrics["confusion_matrix"]

print("=== Validation Metrics ===")
print(f"Accuracy : {metrics['accuracy']:.4f}")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall   : {metrics['recall']:.4f}")
print(f"F1-score : {metrics['f1']:.4f}")
print("\n=== Classification Report ===")
print(metrics["classification_report"])

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Pred 0 (Invalid)", "Pred 1 (Valid)"],
    yticklabels=["True 0 (Invalid)", "True 1 (Valid)"],
)
plt.title("Confusion Matrix - Validation Set")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.show()

In [ ]:
from utils.training import rolling_accuracy

# -----------------------------
# 7) Rolling random accuracy evaluation
# -----------------------------

block_size = 100
num_samples = 50

rolling_acc = rolling_accuracy(
    y_true,
    y_pred,
    block_size=block_size,
    num_samples=num_samples,
    seed=42,
)

print(f"Collected rolling samples: {len(rolling_acc)}")
if len(rolling_acc) == 0:
    print("Block size is larger than evaluation set; reduce block_size.")

In [ ]:
# -----------------------------
# 8) Rolling accuracy distribution
# -----------------------------

if len(rolling_acc) > 0:
    plt.figure(figsize=(8, 5))
    plt.hist(rolling_acc, bins=300, color="skyblue", edgecolor="black")
    plt.xlabel("Accuracy per block")
    plt.ylabel("Number of blocks")
    plt.title(f"Random rolling accuracy histogram (block size={block_size}, samples={num_samples})")
    plt.show()

    print(f"Mean rolling accuracy: {rolling_acc.mean():.4f}")
    print(f"Std rolling accuracy : {rolling_acc.std():.4f}")
else:
    print("No rolling scores to plot. Reduce block_size or increase evaluation samples.")